# Scaling Diagnostic (BACKLOG §1.8 follow-up)

**Date:** 2026-04-18.
**Question:** Is the T-3d high-volume under-prediction a scaling LEVEL problem (KDE shape OK, scaling threshold gates the fix) or a scaling SHAPE problem (KDE density distributed wrong)?

**Method:** For each target at T-3d, log:
- `observed_count`: reviews seen by snap
- `expected_so_far`: KDE integral over the observed window `[snap_dbc, first_review_dbc]` (with ALL critics, matching `_compute_scaling`)
- `obs_over_exp = observed_count / expected_so_far`  (what scaling would WANT to multiply by)
- `scaling_applied`: what `_compute_scaling` actually returns (1.0 if threshold-gated, else clamped `obs_over_exp`)
- `threshold_gated`: True if `expected_so_far < 40`

Stratify by `actual_phase1` quartile. Look for the signal in Q4 (under-predicted).

**Decision logic:**
| Q4 `obs_over_exp` | Q4 `threshold_gated` rate | Interpretation |
|---|---|---|
| > 2 | high | KDE level signal exists, threshold blocks it → **try threshold sweep** |
| ≈ 1 | — | KDE fits observed OK, under-prediction is shape → **scaling can't fix it, Path B** |
| > 2 | low | Scaling firing but clamp binds → (already ruled out by clamp sweep) |

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name != 'notebooks':
    ROOT = ROOT / 'notebooks'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from _helpers import (
    reviews, close_date_map, gaps, gap_lookup, first_review_ts,
    gap_for_slug,
    combined_score_selector,
    snapshot_state, actual_remaining, close_day_count,
    build_critic_profiles, build_kde_lambda_model_capped,
    predict_window_custom, _compute_scaling_custom,
    passes_skip_rules_for_snap,
    CACHE_DIR,
)
from rotten_tomatoes_forecasting.critic_model import _blended_integral

SHIP_ALPHA = 0.5
SHIP_SIGMA_GAP = 8.0
SHIP_N_TRAINING = 20
SHIP_BANDWIDTH_FLOOR = 0.5
SHIP_BANDWIDTH_CEIL = 0.7
SNAP = 3.0  # T-3d is where the issue was surfaced

DIAG_CACHE = CACHE_DIR / 'scaling_diagnostic.pkl'
print('Ready.')

## Run diagnostic at T-3d

In [ ]:
def expected_so_far_all_critics(model, dbc_from, dbc_to):
    """Sum of w_i × integral for ALL critics in training (what _compute_scaling uses)."""
    pop_integral = model.population_prior.integrate_box_1d(dbc_to, dbc_from)
    total = 0.0
    for _, row in model.profiles.df.iterrows():
        w = row['base_rate']
        entry = model.critic_kdes.get(row['reviewer_name'])
        if entry is None:
            continue
        integral = _blended_integral(
            entry, model.population_prior, dbc_to, dbc_from, pop_integral=pop_integral,
        )
        total += w * integral
    return total


def run_diagnostic(force=False):
    if DIAG_CACHE.exists() and not force:
        return pd.read_pickle(DIAG_CACHE)

    rows = []
    for i, target in enumerate(close_date_map):
        target_gap = gap_for_slug(target)
        if target_gap is None:
            continue
        target_close = close_date_map[target]
        midnight_utc_dbc = (target_close - target_close.floor('D')).total_seconds() / 86400
        snap_time = target_close - pd.Timedelta(days=SNAP)
        state = snapshot_state(target, snap_time)
        passed, reason = passes_skip_rules_for_snap(state, SNAP)
        if not passed:
            continue

        target_window_days = state['first_review_dbc'] - SNAP
        target_critics = state['observed_critics']

        training, _ = combined_score_selector(
            target, target_gap, target_critics, target_window_days,
            k=SHIP_N_TRAINING, alpha=SHIP_ALPHA, sigma_gap=SHIP_SIGMA_GAP,
        )
        if len(training) < 5:
            continue

        profiles = build_critic_profiles(reviews, close_date_map, training, verbose=False)
        model = build_kde_lambda_model_capped(
            profiles,
            bandwidth_floor=SHIP_BANDWIDTH_FLOOR,
            bandwidth_ceiling=SHIP_BANDWIDTH_CEIL,
        )

        # Scaling mechanics: observed window = [snap_dbc, first_review_dbc]
        expected_so_far = expected_so_far_all_critics(
            model, dbc_from=state['first_review_dbc'], dbc_to=SNAP,
        )
        obs_over_exp = state['observed_count'] / expected_so_far if expected_so_far > 0 else float('inf')
        threshold_gated = expected_so_far < 40.0
        scaling_applied = _compute_scaling_custom(
            model, days_before_close=SNAP,
            observed_count=state['observed_count'],
            first_review_dbc=state['first_review_dbc'],
        )

        # Phase 1 prediction with ship-default scaling
        phase1 = predict_window_custom(
            model, dbc_from=SNAP, dbc_to=midnight_utc_dbc,
            observed_critics=target_critics,
            observed_count=state['observed_count'],
            first_review_dbc=state['first_review_dbc'],
        )
        # Phase 1 prediction with scaling DISABLED (clamp=(1,1) effectively, but threshold still returns 1.0)
        # Just pass observed_count=None to skip scaling entirely:
        phase1_unscaled = predict_window_custom(
            model, dbc_from=SNAP, dbc_to=midnight_utc_dbc,
            observed_critics=target_critics,
            observed_count=None, first_review_dbc=None,
        )

        # Actual phase 1
        movie_reviews = reviews[reviews['movie_slug'] == target].copy()
        movie_reviews['dbc'] = (target_close - movie_reviews['estimated_timestamp']).dt.total_seconds() / 86400
        actual_p1 = int(((movie_reviews['dbc'] > midnight_utc_dbc) & (movie_reviews['dbc'] <= SNAP)).sum())

        rows.append({
            'target': target,
            'observed_count': state['observed_count'],
            'first_review_dbc': state['first_review_dbc'],
            'expected_so_far': expected_so_far,
            'obs_over_exp': obs_over_exp,
            'threshold_gated': threshold_gated,
            'scaling_applied': scaling_applied,
            'phase1_pred_scaled': float(phase1),
            'phase1_pred_unscaled': float(phase1_unscaled),
            'actual_phase1': actual_p1,
            'err_scaled': float(phase1) - actual_p1,
        })
        if (i + 1) % 30 == 0:
            print(f'  {i+1}/{len(close_date_map)} targets')

    df = pd.DataFrame(rows)
    df.to_pickle(DIAG_CACHE)
    print(f'Cached {len(df)} rows to {DIAG_CACHE.name}')
    return df

diag = run_diagnostic()
print(f'\nn = {len(diag)}')
print(diag.describe().round(2).to_string())

## Stratify by actual_phase1 quartile

In [ ]:
diag['q_actual'] = pd.qcut(diag['actual_phase1'], q=4, labels=['Q1','Q2','Q3','Q4'], duplicates='drop')

print('Scaling state by actual_phase1 quartile:')
print()
strat = []
for q in ['Q1','Q2','Q3','Q4']:
    sub = diag[diag['q_actual'] == q]
    if not len(sub):
        continue
    strat.append({
        'quartile': q,
        'n': len(sub),
        'actual_p1_range': f'{int(sub["actual_phase1"].min())}-{int(sub["actual_phase1"].max())}',
        'median_observed': int(sub['observed_count'].median()),
        'median_expected': round(sub['expected_so_far'].median(), 1),
        'median_obs_over_exp': round(sub['obs_over_exp'].median(), 2),
        'p90_obs_over_exp': round(sub['obs_over_exp'].quantile(0.9), 2),
        'pct_threshold_gated': round(sub['threshold_gated'].mean() * 100, 1),
        'median_scaling_applied': round(sub['scaling_applied'].median(), 2),
        'pct_clamp_upper': round((sub['scaling_applied'] >= 1.99).mean() * 100, 1),
        'mean_err_scaled': round(sub['err_scaled'].mean(), 2),
    })
print(pd.DataFrame(strat).to_string(index=False))

## H/m subset (under-predicted targets)

Drill into the 5 movies flagged by f_audit.

In [ ]:
HM = ['the_drama', 'the_super_mario_galaxy_movie', 'forbidden_fruits_2026',
      'they_will_kill_you', 'you_me_and_tuscany']
hm_diag = diag[diag['target'].isin(HM)].copy()
print('H/m subset diagnostic at T-3d:')
cols = ['target', 'observed_count', 'first_review_dbc', 'expected_so_far',
        'obs_over_exp', 'threshold_gated', 'scaling_applied',
        'actual_phase1', 'phase1_pred_scaled']
print(hm_diag[cols].to_string(index=False, float_format='%.2f'))

## Interpretation

Read the Q4 row above alongside the h/m subset:

- **If Q4 median `obs_over_exp` > 2 AND `pct_threshold_gated` > 50%:** the level signal is there, threshold blocks it. Threshold sweep is warranted.
- **If Q4 median `obs_over_exp` ≈ 1:** KDE fits observed window OK; under-prediction is shape. Threshold/clamp can't fix it. Path B (volume feature).
- **If Q4 median `obs_over_exp` > 2 AND `pct_clamp_upper` > 50%:** clamp binding — but we already ruled this out.